# Sesión 5 — Indexación avanzada y selección de datos
## Filtros complejos, `.query()` y selección profesional

**Contexto clínico:** Perfiles clínicos caninos — selección de casos para análisis  
**Herramientas:** Python 3.10 · pandas · NumPy

In [ ]:
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

# Reconstruir el dataset completo
# A partir de la próxima sesión cargaremos siempre desde CSV
# Hoy lo reconstruimos para que el notebook sea autónomo

datos = {
    'paciente_id':         ['C001','C002','C003','C004','C005',
                            'C006','C007','C008','C009','C010',
                            'C011','C012','C013','C014','C015'],
    'raza':                ['Labrador','Beagle','Pastor Alemán','Bulldog','Golden',
                            'Schnauzer','Poodle','Boxer','Dálmata','Husky',
                            'Chihuahua','Rottweiler','Cocker','Doberman','Shih Tzu'],
    'edad_años':           [3, 7, 5, 4, 2, 9, 6, 8, 3, 5, 4, 6, 11, 7, 2],
    'peso_kg':             [28.5,12.0,35.2,22.1,30.0,8.5,6.2,29.8,24.5,27.3,
                            3.1,45.2,11.8,38.4,5.9],
    'sexo':                ['M','H','M','M','H','H','M','H','M','H',
                            'H','M','H','M','H'],
    'especie':             ['Canino'] * 15,

    # Serie roja
    'eritrocitos_mill_uL': [6.2,4.8,7.1,3.9,6.5,5.2,5.8,4.1,6.8,5.5,
                            6.0,7.2,3.5,6.6,5.9],
    'hemoglobina_g_dL':    [15.1,11.8,17.2, 9.5,16.0,12.5,14.2,10.2,16.5,13.8,
                            14.8,17.8, 8.9,16.2,14.5],
    'hematocrito_pct':     [45.2,35.4,51.8,28.6,48.0,37.5,42.6,30.8,49.5,41.4,
                            44.5,53.5,26.8,48.6,43.5],
    'VCM_fL':              [72.9,73.8,72.9,73.3,73.8,72.1,73.4,75.1,72.8,75.3,
                            74.2,74.3,76.6,73.6,73.7],
    'HCM_pg':              [24.4,24.6,24.2,24.4,24.6,24.0,24.5,24.9,24.3,25.1,
                            24.7,24.7,25.4,24.5,24.6],
    'CHCM_g_dL':           [33.4,33.3,33.2,33.2,33.3,33.3,33.3,33.1,33.3,33.3,
                            33.3,33.2,33.2,33.4,33.3],

    # Serie blanca
    'leucocitos_mil_uL':   [8.5,12.3,7.2,18.6,9.1,11.4,8.8,15.2,7.9,10.3,
                            9.2,8.1,22.4,7.5,10.8],
    'neutrofilos_pct':     [68,75,65,82,70,78,67,80,66,72,69,64,85,67,71],
    'linfocitos_pct':      [22,16,25,10,20,14,24,12,25,18,21,27, 8,24,20],
    'monocitos_pct':       [ 6, 5, 6, 5, 6, 5, 6, 5, 6, 6, 7, 6, 5, 6, 6],
    'eosinofilos_pct':     [ 3, 3, 3, 2, 3, 2, 2, 2, 2, 3, 2, 2, 1, 2, 2],
    'basofilos_pct':       [ 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],

    # Plaquetas
    'plaquetas_mil_uL':    [245,189,310, 98,278,220,265,142,290,235,
                            255,320, 88,275,248],

    # Bioquímica
    'glucosa_mg_dL':       [ 95,102, 88,145, 91,110, 87,132, 93, 98,
                             101, 89,155, 96,108],
    'bun_mg_dL':           [ 18, 22, 15, 45, 19, 28, 16, 38, 17, 21,
                              20, 14, 52, 18, 23],
    'creatinina_mg_dL':    [0.9,1.1,0.8,2.8,1.0,1.3,0.9,2.1,0.8,1.0,
                            1.1,0.7,3.2,0.9,1.2],
    'alt_U_L':             [ 32, 45, 28, 89, 35, 67, 30,112, 29, 41,
                              38, 25,125, 33, 48],
    'fosfatasa_alk_U_L':   [ 45, 88, 40,155, 52,210, 38, 98, 42, 65,
                              72, 35,188, 48, 91],
    'proteinas_totales_g_dL': [6.8,7.1,6.5,5.9,7.0,6.2,6.9,5.8,7.2,6.7,
                                6.6,7.3,5.5,6.8,7.0],

    'diagnostico':         ['Normal','Anemia leve','Normal',
                            'Anemia moderada + AZO','Normal',
                            'Normal','Normal','Anemia leve + AZO',
                            'Normal','Normal',
                            'Normal','Normal','Anemia moderada + AZO + Leucocitosis',
                            'Normal','Normal']
}

df = pd.DataFrame(datos)
print(f"Dataset: {df.shape[0]} pacientes × {df.shape[1]} variables")
df.head(3)

## 1. Filtros booleanos complejos — más allá de una sola condición

La regla práctica: cuanto más compleja la condición, más importante es  
estructurarla en variables intermedias con nombres clínicamente descriptivos.

In [ ]:
# Patrón profesional: definir cada condición como variable nombrada
# Esto hace el código legible para cualquier colega —clínico o analista—
# y facilita enormemente el debugging

# Criterios de la serie roja
anemia          = df['hematocrito_pct']    < 37.0
anemia_severa   = df['hematocrito_pct']    < 25.0
anemia_moderada = (df['hematocrito_pct'] >= 25.0) & (df['hematocrito_pct'] < 37.0)

# Criterios de la serie blanca
leucocitosis    = df['leucocitos_mil_uL']  > 17.0
leucopenia      = df['leucocitos_mil_uL']  < 6.0
neutrofilia     = df['neutrofilos_pct']    > 77.0

# Criterios bioquímicos
azotemia        = df['creatinina_mg_dL']   > 1.5
hepatopatia     = df['alt_U_L']            > 80.0
hiperglucemia   = df['glucosa_mg_dL']      > 120.0

# Criterios de plaquetas
trombocitopenia = df['plaquetas_mil_uL']   < 200.0
trombocitopenia_severa = df['plaquetas_mil_uL'] < 100.0

print("Condiciones clínicas definidas ✓")
print(f"Pacientes con anemia:          {anemia.sum()}")
print(f"Pacientes con leucocitosis:    {leucocitosis.sum()}")
print(f"Pacientes con azotemia:        {azotemia.sum()}")
print(f"Pacientes con trombocitopenia: {trombocitopenia.sum()}")

In [ ]:
# Caso clínico 1: síndrome paraneoplásico sospechoso
# Anemia + trombocitopenia + leucocitosis en el mismo paciente
# Esta combinación en un perro adulto es una bandera seria

sindrome_paraneoplasico = anemia & trombocitopenia & leucocitosis

cols = ['paciente_id', 'raza', 'edad_años',
        'hematocrito_pct', 'plaquetas_mil_uL', 
        'leucocitos_mil_uL', 'diagnostico']

print("Pacientes con patrón: anemia + trombocitopenia + leucocitosis")
resultado = df.loc[sindrome_paraneoplasico, cols]

if len(resultado) > 0:
    print(resultado.to_string(index=False))
else:
    print("Ningún paciente cumple los tres criterios simultáneamente")

In [ ]:
# Caso clínico 2: perfil de enfermedad renal crónica (ERC)
# Azotemia + hipoproteinemia + anemia normocítica normocrómica
# (El patrón clásico de ERC canina)

vcm_normal  = df['VCM_fL'].between(60.0, 77.0)     # .between() — lo veremos en detalle
chcm_normal = df['CHCM_g_dL'].between(32.0, 36.0)
hipoproteinemia = df['proteinas_totales_g_dL'] < 6.0

perfil_erc = azotemia & anemia & vcm_normal & chcm_normal

print("Perfil compatible con ERC (azotemia + anemia normocítica + normocrómica):")
cols_erc = ['paciente_id', 'raza', 'edad_años', 'creatinina_mg_dL',
            'bun_mg_dL', 'hematocrito_pct', 'VCM_fL', 'CHCM_g_dL']
print(df.loc[perfil_erc, cols_erc].to_string(index=False))

In [ ]:
# Caso clínico 3: perfil de proceso infeccioso bacteriano agudo
# Leucocitosis + neutrofilia + proteínas totales elevadas (respuesta de fase aguda)

proteinas_elevadas = df['proteinas_totales_g_dL'] > 7.0
infeccion_bacteriana = leucocitosis & neutrofilia & proteinas_elevadas

print("Perfil compatible con infección bacteriana aguda:")
cols_inf = ['paciente_id', 'raza', 'leucocitos_mil_uL', 
            'neutrofilos_pct', 'proteinas_totales_g_dL', 'diagnostico']
resultado_inf = df.loc[infeccion_bacteriana, cols_inf]

if len(resultado_inf) > 0:
    print(resultado_inf.to_string(index=False))
else:
    print("Ningún paciente cumple este perfil en el dataset actual")

# Mostrar cuántos cumplen cada criterio individualmente
print(f"\nCriterios individuales:")
print(f"  Leucocitosis:          {leucocitosis.sum()} pacientes")
print(f"  Neutrofilia:           {neutrofilia.sum()} pacientes")
print(f"  Proteínas elevadas:    {proteinas_elevadas.sum()} pacientes")
print(f"  Los tres simultáneos:  {infeccion_bacteriana.sum()} pacientes")

## 2. Métodos especializados de filtrado

pandas tiene métodos específicos que hacen el código más legible  
que los filtros booleanos equivalentes: `.isin()`, `.between()`, `.str`.

In [ ]:
# .isin() — equivale al operador IN de SQL
# Más limpio que múltiples condiciones con |

# Seleccionar razas con predisposición a ERC
razas_predispuestas_erc = ['Cocker', 'Bulldog', 'Shih Tzu', 'Doberman']

filtro_razas = df['raza'].isin(razas_predispuestas_erc)

print(f"Razas con predisposición a ERC en el dataset: {filtro_razas.sum()} pacientes")
print(df.loc[filtro_razas, ['paciente_id', 'raza', 'creatinina_mg_dL', 
                              'bun_mg_dL', 'diagnostico']].to_string(index=False))

In [ ]:
# ~isin() — el complemento: registros que NO están en la lista
# El operador ~ invierte una Serie booleana (equivale a NOT en SQL)

razas_excluidas = ['Chihuahua', 'Poodle', 'Shih Tzu']
perros_medianos_grandes = ~df['raza'].isin(razas_excluidas)

print(f"Pacientes (excluyendo razas pequeñas): {perros_medianos_grandes.sum()}")
print(df.loc[perros_medianos_grandes, ['paciente_id', 'raza', 'peso_kg']]\
        .sort_values('peso_kg', ascending=False)\
        .to_string(index=False))

In [ ]:
# .between(min, max) — equivale a: (col >= min) & (col <= max)
# Ambos extremos son INCLUSIVOS por defecto

# Rango de referencia de creatinina canina: 0.5–1.5 mg/dL
creatinina_normal = df['creatinina_mg_dL'].between(0.5, 1.5)
print(f"Creatinina en rango normal: {creatinina_normal.sum()} de {len(df)} pacientes")

# Pacientes en rango normal de creatinina Y con BUN elevado
# Esto podría indicar azotemia pre-renal leve o estado de deshidratación
bun_elevado = df['bun_mg_dL'] > 25
prerrenal_posible = creatinina_normal & bun_elevado

print(f"\nPosible azotemia pre-renal (creatinina normal + BUN alto):")
cols_prer = ['paciente_id', 'raza', 'bun_mg_dL', 
             'creatinina_mg_dL', 'proteinas_totales_g_dL']
print(df.loc[prerrenal_posible, cols_prer].to_string(index=False))

In [ ]:
# Combinación de .between() para múltiples analitos
# Pacientes con TODOS los índices eritrocitarios dentro del rango normal
# (VCM, HCM, CHCM — útil para confirmar anemia normocítica normocrómica)

vcm_normal  = df['VCM_fL'].between(60.0, 77.0)
hcm_normal  = df['HCM_pg'].between(19.5, 24.5)
chcm_normal = df['CHCM_g_dL'].between(32.0, 36.0)

indices_normales = vcm_normal & hcm_normal & chcm_normal

print(f"Pacientes con índices eritrocitarios todos normales: {indices_normales.sum()}")
print(f"Pacientes con algún índice alterado: {(~indices_normales).sum()}")

In [ ]:
# El accessor .str permite aplicar operaciones de string a una columna completa
# Equivale a llamar métodos de String de Java sobre cada elemento, pero vectorizado

# .str.contains() — buscar substring en columna de texto
# Útil para buscar en diagnósticos, notas clínicas, nombres de patógenos

tiene_azo = df['diagnostico'].str.contains('AZO', na=False)
print("Pacientes con azotemia en el diagnóstico:")
print(df.loc[tiene_azo, ['paciente_id', 'diagnostico']].to_string(index=False))

In [ ]:
# .str.startswith() y .str.endswith()
diagnosticos_anemia = df['diagnostico'].str.startswith('Anemia')
print(f"Diagnósticos que empiezan con 'Anemia': {diagnosticos_anemia.sum()}")

# .str.upper(), .str.lower(), .str.strip() — normalización de texto
# Crítico cuando los datos vienen de sistemas con inconsistencias de capitalización
razas_upper = df['raza'].str.upper()
print(f"\nRazas en mayúsculas (primeras 5): {razas_upper.head().tolist()}")

# .str.len() — longitud del string por elemento
print(f"\nLongitud del diagnóstico más largo: {df['diagnostico'].str.len().max()} chars")
print(f"Diagnóstico más largo: {df.loc[df['diagnostico'].str.len().idxmax(), 'diagnostico']}")

In [ ]:
# .str.split() — dividir strings
# Útil para parsear campos combinados (ej: "Anemia moderada + AZO")

# Separar diagnósticos compuestos en lista de condiciones
df['condiciones_lista'] = df['diagnostico'].str.split(' + ')

print("Diagnósticos separados en lista:")
for _, fila in df[['paciente_id', 'condiciones_lista']].iterrows():
    if len(fila['condiciones_lista']) > 1:
        print(f"  {fila['paciente_id']}: {fila['condiciones_lista']}")

# Contar cuántas condiciones tiene cada paciente
df['n_condiciones'] = df['condiciones_lista'].str.len()
print(f"\nDistribución de número de condiciones por paciente:")
print(df['n_condiciones'].value_counts().sort_index())

# Limpiar: eliminar columna auxiliar que no necesitaremos más
df = df.drop(columns=['condiciones_lista'])

## 3. `.query()` — filtros con sintaxis tipo SQL

`.query()` permite escribir condiciones como strings de texto legible.  
Es especialmente útil para condiciones complejas o cuando el código se comparte  
con personas que vienen de SQL (muy común en equipos de laboratorio).

In [ ]:
# Sintaxis básica: df.query("condición como string")
# Los nombres de columnas van sin comillas dentro del string
# Los strings de datos van con comillas dentro del string

# Equivalente a: df[df['creatinina_mg_dL'] > 1.5]
azotemia_query = df.query("creatinina_mg_dL > 1.5")
print(f"Pacientes con azotemia (query): {len(azotemia_query)}")
print(azotemia_query[['paciente_id', 'creatinina_mg_dL', 'bun_mg_dL']].to_string(index=False))

In [ ]:
# Condiciones múltiples — and/or en lugar de &/|
# La sintaxis de .query() acepta 'and', 'or', 'not' (más legible)

perfil_erc_query = df.query(
    "creatinina_mg_dL > 1.5 and "
    "hematocrito_pct < 37.0 and "
    "VCM_fL >= 60.0 and VCM_fL <= 77.0"
)
print("Perfil ERC con .query():")
print(perfil_erc_query[['paciente_id', 'creatinina_mg_dL', 
                          'hematocrito_pct', 'VCM_fL']].to_string(index=False))

In [ ]:
# .query() con variables externas — usar @ para referenciar variables de Python
limite_creatinina = 1.5
limite_hto        = 37.0

# El prefijo @ indica que es una variable Python, no una columna
resultado = df.query("creatinina_mg_dL > @limite_creatinina and hematocrito_pct < @limite_hto")
print(f"Usando variables externas en .query(): {len(resultado)} pacientes")

In [ ]:
# .query() con .isin() — sintaxis diferente
razas_interes = ['Cocker', 'Bulldog', 'Labrador']

# En .query() las listas se referencian con @
resultado_razas = df.query("raza in @razas_interes")
print(f"Razas de interés: {resultado_razas['raza'].tolist()}")

# Comparación directa: .query() vs filtro booleano
# Para condiciones simples, ambos son equivalentes
# .query() es más legible; filtro booleano es más flexible para casos complejos

## 4. Selección dinámica de columnas

En análisis real no siempre sabes de antemano qué columnas necesitas.  
pandas tiene herramientas para seleccionar columnas por patrón, tipo o lista dinámica.

In [ ]:
# Seleccionar columnas que contienen un substring en el nombre
# Útil cuando tienes decenas de columnas y quieres las de un panel específico

# Todas las columnas de la serie blanca (contienen 'pct' o 'leucocitos')
cols_serie_blanca = [col for col in df.columns 
                     if 'pct' in col or 'leucocitos' in col]
print(f"Columnas de serie blanca: {cols_serie_blanca}")
print(df[cols_serie_blanca].head(3))

In [ ]:
# .filter() — selección de columnas por patrón
# like: contiene el substring | regex: expresión regular

# Columnas de bioquímica (contienen '_g_dL' o '_mg_dL' o '_U_L')
cols_bioquimica = df.filter(like='_dL').columns.tolist()
print(f"Columnas con '_dL' en el nombre: {cols_bioquimica}")

# Columnas numéricas — select_dtypes()
cols_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nColumnas numéricas ({len(cols_numericas)}):")
print(cols_numericas)

# Columnas de texto (object en pandas = string)
cols_texto = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nColumnas de texto ({len(cols_texto)}):")
print(cols_texto)

In [ ]:
# Patrón muy útil: construir la lista de columnas del reporte dinámicamente
# según los criterios de análisis del momento

def construir_reporte(df, condicion, analitos, incluir_diagnostico=True):
    """
    Genera un reporte filtrado dinámicamente.
    
    Parámetros:
        df              : DataFrame base
        condicion       : Serie booleana de filtrado
        analitos        : lista de columnas de analitos a incluir
        incluir_diagnostico: bool — agregar columna de diagnóstico
    
    Retorna:
        DataFrame filtrado y ordenado
    """
    cols_base = ['paciente_id', 'raza', 'edad_años']
    
    if incluir_diagnostico:
        cols_reporte = cols_base + analitos + ['diagnostico']
    else:
        cols_reporte = cols_base + analitos
    
    # Filtrar solo columnas que existen (protección contra errores)
    cols_validas = [c for c in cols_reporte if c in df.columns]
    
    return df.loc[condicion, cols_validas].reset_index(drop=True)


# Usar la función con distintos criterios
print("=== REPORTE: PACIENTES CON AZOTEMIA ===")
reporte_azotemia = construir_reporte(
    df,
    condicion = df['creatinina_mg_dL'] > 1.5,
    analitos  = ['bun_mg_dL', 'creatinina_mg_dL', 'proteinas_totales_g_dL']
)
print(reporte_azotemia.to_string(index=False))

print("\n=== REPORTE: PACIENTES CON LEUCOCITOSIS ===")
reporte_leuco = construir_reporte(
    df,
    condicion = df['leucocitos_mil_uL'] > 17.0,
    analitos  = ['leucocitos_mil_uL', 'neutrofilos_pct', 'linfocitos_pct']
)
print(reporte_leuco.to_string(index=False))

## 5. Encadenamiento — operaciones en cadena limpia

pandas permite encadenar múltiples operaciones con el operador `.`  
Cuando se hace bien, el código se lee casi como lenguaje natural.  
Cuando se hace mal, genera errores difíciles de depurar.

In [ ]:
# Encadenamiento básico — cada método devuelve un DataFrame
# sobre el que se puede llamar al siguiente método

resumen_alterados = (
    df
    .query("hematocrito_pct < 37.0 or creatinina_mg_dL > 1.5 or leucocitos_mil_uL > 17.0")
    .loc[:, ['paciente_id', 'raza', 'hematocrito_pct', 
              'creatinina_mg_dL', 'leucocitos_mil_uL', 'diagnostico']]
    .sort_values('creatinina_mg_dL', ascending=False)
    .reset_index(drop=True)
)

print(f"Pacientes con al menos una alteración mayor: {len(resumen_alterados)}")
print(resumen_alterados.to_string(index=False))

In [ ]:
# Encadenamiento con transformación y agregación
# Pregunta clínica: ¿cuál es el perfil bioquímico promedio de los pacientes anémicos
# vs los no anémicos, estratificado por sexo?

perfil_por_estado = (
    df
    .assign(es_anemico = df['hematocrito_pct'] < 37.0)   # .assign() agrega columna sin modificar df
    .groupby(['es_anemico', 'sexo'])[['creatinina_mg_dL', 'bun_mg_dL', 'alt_U_L']]
    .mean()
    .round(2)
)

print("Perfil bioquímico promedio por estado de anemia y sexo:")
print(perfil_por_estado)

In [ ]:
# .assign() es la forma correcta de agregar columnas en una cadena
# A diferencia de df['col'] = ..., no modifica el DataFrame original

df_enriquecido = (
    df
    .assign(
        HCM_calculado = lambda x: (x['hemoglobina_g_dL'] / x['eritrocitos_mill_uL']) * 10,
        ratio_bun_cr  = lambda x: x['bun_mg_dL'] / x['creatinina_mg_dL'],
        n_alertas     = lambda x: (
            (x['hematocrito_pct']    < 37.0).astype(int) +
            (x['creatinina_mg_dL']   > 1.5 ).astype(int) +
            (x['leucocitos_mil_uL']  > 17.0).astype(int) +
            (x['plaquetas_mil_uL']   < 200.0).astype(int)
        )
    )
    .sort_values('n_alertas', ascending=False)
)

print("DataFrame enriquecido (primeros 5, por número de alertas):")
cols_ver = ['paciente_id', 'HCM_calculado', 'ratio_bun_cr', 'n_alertas', 'diagnostico']
print(df_enriquecido[cols_ver].head().round(2).to_string(index=False))

In [ ]:
# === REPORTE A — PANEL RENAL ===
bun_elevado        = df['bun_mg_dL']        > 25.0
creatinina_elevada = df['creatinina_mg_dL'] > 1.2

panel_renal_filtro = bun_elevado | creatinina_elevada

cols_renal = ['paciente_id', 'raza', 'edad_años', 
              'bun_mg_dL', 'creatinina_mg_dL', 
              'proteinas_totales_g_dL', 'diagnostico']

reporte_renal = (
    df
    .loc[panel_renal_filtro, cols_renal]
    .sort_values('creatinina_mg_dL', ascending=False)
    .reset_index(drop=True)
)

print("=" * 60)
print("REPORTE A — PANEL RENAL")
print(f"Pacientes incluidos: {len(reporte_renal)}")
print("=" * 60)
print(reporte_renal.to_string(index=False))


# === REPORTE B — PANEL HEPÁTICO ===
alt_elevada        = df['alt_U_L']           > 60.0
fosf_elevada       = df['fosfatasa_alk_U_L'] > 150.0
glucosa_elevada    = df['glucosa_mg_dL']      > 100.0

# Condición compuesta: (ALT alta O fosfatasa alta) Y glucosa alta
panel_hepatico_filtro = (alt_elevada | fosf_elevada) & glucosa_elevada

cols_hepatico = ['paciente_id', 'raza', 
                 'alt_U_L', 'fosfatasa_alk_U_L', 
                 'glucosa_mg_dL', 'diagnostico']

reporte_hepatico = (
    df
    .loc[panel_hepatico_filtro, cols_hepatico]
    .assign(ratio_alt_fosf = lambda x: (x['alt_U_L'] / x['fosfatasa_alk_U_L']).round(3))
    .reset_index(drop=True)
)

print("\n" + "=" * 60)
print("REPORTE B — PANEL HEPÁTICO")
print(f"Pacientes incluidos: {len(reporte_hepatico)}")
print("=" * 60)
print(reporte_hepatico.to_string(index=False))

print(f"\nResumen ejecutivo:")
print(f"  Panel renal:   {len(reporte_renal)} pacientes")
print(f"  Panel hepático: {len(reporte_hepatico)} pacientes")